# OCR demo — library & archival materials

A walkthrough of the extraction pipeline tuned for digitized library
content: books, manuscripts, sheet music, newspapers, maps,
correspondence, ephemera, multilingual scans.

The pipeline mirrors the grant-admin demo but with two differences that
matter for archival material:

- **Page-at-a-time, not chunked.** Each page goes through independently
  with a 3-page sliding window for cross-page continuation. We don't
  batch like the grant pipeline does, because faithful verbatim
  transcription is the primary goal — and that's a per-page concern.
- **Different schema.** Body-text transcription, bibliographic metadata,
  marginalia, stamps/bookplates, musical notation, page type, physical
  condition. No "stakeholders" or "tables-as-budget-grids".

Pass 2 then synthesizes catalog-style metadata across pages (title,
type, creator, date, subject tags, summary).

> **Grant-admin docs?** Use `RSP_example_extraction_pipeline_demo.ipynb` instead.


## 1. Connect to the deployed vLLM endpoint

The model runs as a separate RunAI workload exposing an OpenAI-compatible
HTTP endpoint. This notebook is a thin client that talks to it.

The cell below picks the right URL based on **`RUN_LOCATION`** at the top
— `in_cluster` for a RunAI workspace, `laptop_vpn` for a laptop on the
campus VPN — auto-detects the served model, and runs a smoke test.


In [ ]:
import os
import httpx

# ── Pick where you're running ─────────────────────────────────────────
# "in_cluster" — RunAI workspace; calls the cluster-internal service hostname
# "laptop_vpn" — laptop on the campus VPN; calls the public Knative hostname
# Set the VLLM_BASE_URL env var to override either default.
#
# Note: "laptop_vpn" goes through the Knative ingress, which has a ~1 MB
# request-body cap. The notebook automatically downscales + re-encodes
# page images as JPEG on this path so the body fits; in-cluster runs send
# full-resolution PNGs since they bypass the ingress.
RUN_LOCATION = "laptop_vpn"

URLS = {
    "in_cluster": "http://qwen3--vl--32b--instruct-awq.runai-shared-models.svc.cluster.local/v1",
    "laptop_vpn": "https://qwen3--vl--32b--instruct-awq-runai-shared-models.deepthought.doit.wisc.edu/v1",
}

VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", URLS[RUN_LOCATION])
# Per-call output budget. The grant-admin demo bumps this to fit dense 10–20
# page chunks; the library demo runs page-by-page so a smaller budget is fine.
VLLM_MAX_TOKENS = 4096
MODEL_NAME = os.environ.get("VLM_MODEL", "QuantTrio/Qwen3-VL-32B-Instruct-AWQ")

# Auto-detect the served model + run a smoke test. Wrapped in try/except so
# a failure here (wrong RUN_LOCATION, VPN off, server down) prints cleanly
# but leaves VLLM_BASE_URL / MODEL_NAME / VLLM_MAX_TOKENS defined for the
# rest of the notebook.
try:
    resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=10)
    resp.raise_for_status()
    MODEL_NAME = resp.json()["data"][0]["id"]
    print(f"Mode:     {RUN_LOCATION}")
    print(f"Endpoint: {VLLM_BASE_URL}")
    print(f"Model:    {MODEL_NAME}")
    print(f"HF card:  https://huggingface.co/{MODEL_NAME}")

    resp = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": "In a single sentence, share with me the entire life story of Winnie the Pooh."}],
            "max_tokens": 100,
            "temperature": 0.0,
        },
        timeout=60,
    )
    resp.raise_for_status()
    print("Smoke test:", resp.json()["choices"][0]["message"]["content"].strip())
except Exception as e:
    print(f"Endpoint unreachable: {type(e).__name__}: {e}")
    print(f"Defaults still set: VLLM_BASE_URL={VLLM_BASE_URL!r}, MODEL_NAME={MODEL_NAME!r}")
    print("Switch RUN_LOCATION above (or set VLLM_BASE_URL env var) and re-run.")


### Running locally?

If you're on the laptop / VPN path, set `RUN_LOCATION = "laptop_vpn"` in
the cell above. A fresh laptop env also won't have the imports this
notebook uses (`httpx`, `pymupdf` / `fitz`, `Pillow`). Quick setup with
[uv](https://docs.astral.sh/uv/) — last line registers the venv as a
JupyterLab kernel named **OCR app (uv venv)** so this notebook
auto-selects it on open:

```bash
uv venv && source .venv/bin/activate
uv pip install httpx pymupdf Pillow ipykernel
python -m ipykernel install --user --name=ocr-app --display-name="OCR app (uv venv)"
```

(or the equivalent `python -m venv` + `pip install` if you're not on uv.)

The endpoint is OpenAI-compatible — `httpx` is what `run_vlm` uses below,
but the `openai` Python client works against the same URL if you prefer.
See [`docs/deploy-vllm.md` → Access from outside the cluster (VPN)](../docs/deploy-vllm.md#access-from-outside-the-cluster-vpn)
for details on the public hostname pattern.


## 2. Notebook-side wrappers

`run_vlm` and `parse_vlm_json` are the same minimal wrappers used in the
grant-admin demo. Library content is more likely to contain literal quote
marks (e.g. titles in quotes inside a transcription), so we use vLLM's
guided JSON output to dodge most of the parsing edge cases up front.


In [ ]:
import base64, io, json, time
from pathlib import Path
from PIL import Image


# Public-ingress request-body cap (laptop_vpn path). The Knative gateway
# limits chat-completions POST bodies to ~1 MB; lossless PNGs of high-DPI
# page images blow past that even after a single downscale. On laptop_vpn
# we both shrink the image AND switch to JPEG (5–10x smaller for typical
# document pages with no perceptible quality loss for text). In-cluster
# calls bypass the ingress, so we keep PNG there for lossless fidelity.
_LAPTOP_VPN_MAX_IMAGE_DIM = 900   # px on the longest edge
_LAPTOP_VPN_JPEG_QUALITY  = 85    # visually indistinguishable from lossless for text


def _encode_image_b64(image: Image.Image) -> str:
    """Serialize a PIL image as a base64 image data URI for the VLM payload.

    The VLM endpoint takes images via OpenAI's `image_url` field. On
    in_cluster runs we send PNG as-is. On laptop_vpn runs we downscale +
    re-encode as JPEG so the request body fits under the public ingress's
    ~1 MB cap.
    """
    if RUN_LOCATION == "laptop_vpn":
        if max(image.size) > _LAPTOP_VPN_MAX_IMAGE_DIM:
            scale = _LAPTOP_VPN_MAX_IMAGE_DIM / max(image.size)
            image = image.resize(
                (int(image.width * scale), int(image.height * scale)),
                Image.LANCZOS,
            )
        buf = io.BytesIO()
        image.convert("RGB").save(
            buf, format="JPEG",
            quality=_LAPTOP_VPN_JPEG_QUALITY, optimize=True,
        )
        return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}"

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"


def run_vlm(messages: list, max_tokens: int = VLLM_MAX_TOKENS) -> str:
    """Send chat messages to vLLM and return the response text.

    Role: HTTP client wrapper for library extraction. Unlike the
    grant-admin demo this version does NOT stream — pages are smaller
    here, max_tokens is much lower (4096 vs 90000), and individual
    calls finish well inside the gateway's idle timeout, so the
    simpler one-shot request is fine. Adds light frequency /
    repetition penalties so the model can break out of generation
    loops on decorative or stylized text, and asks for guided JSON
    at the token level.
    """
    oai_messages = []
    for msg in messages:
        content = []
        for part in msg["content"]:
            if part["type"] == "text":
                content.append({"type": "text", "text": part["text"]})
            elif part["type"] == "image":
                content.append({"type": "image_url", "image_url": {"url": part["image"]}})
        oai_messages.append({"role": msg["role"], "content": content})

    payload = {
        "model": MODEL_NAME,
        "messages": oai_messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        # VLMs sometimes loop on decorative/stylized text. Light penalties
        # push the model out of those loops without hurting transcription.
        "frequency_penalty": 0.3,
        "repetition_penalty": 1.05,
        "response_format": {"type": "json_object"},
    }
    resp = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=300,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


def parse_vlm_json(raw: str):
    """Turn a raw VLM response string into a Python dict.

    Same role as the grant-admin demo: strip optional markdown fences,
    try strict parsing, fall back to the outermost balanced `{...}`.
    Returns ``(parsed_dict, None)`` on success or ``(None, error_msg)``
    on failure.
    """
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        first, last = cleaned.find("{"), cleaned.rfind("}")
        if first != -1 and last > first:
            try:
                return json.loads(cleaned[first:last + 1]), None
            except json.JSONDecodeError:
                pass
        return None, str(e)


def extract_links(page) -> list:
    """Pull hyperlinks from a PDF page's metadata layer.

    Role: cheap (text, URL) pairs we can drop into the prompt as extra
    context. Library scans usually have none (image-only PDFs), but
    modern digitized newspapers, indexes, and finding aids sometimes
    embed live links — surfacing them lets the VLM tie URLs back to
    the surrounding text it transcribes.
    """
    links = []
    for link in page.get_links():
        uri = link.get("uri", "")
        if not uri:
            continue
        rect = link.get("from", None)
        text = page.get_text("text", clip=rect).strip() if rect else ""
        links.append({"text": text, "url": uri})
    return links


print("Wrappers ready.")


## 3. The library extraction prompt

The prompt is built around the rule: **body_text always wins**. The VLM
is told that faithful verbatim transcription is the primary task and
everything else (bibliographic metadata, page type, marginalia detection)
is secondary. Without that framing the model treats stylized title-page
text as "decoration" and silently skips it.

The schema is library/archive-flavored:

- `body_text` — verbatim transcription of the page
- `page_type` — book_page, manuscript, sheet_music, etc.
- `bibliographic` — title/creator/date/language/publisher/place
- `structure` — headings, footnotes, lists, poetry/verse
- `visual_elements` — illustrations, musical notation, maps
- `marginalia`, `stamps_marks`, `physical_condition`
- `continuation` — flags for content spilling across pages


In [ ]:
LIBRARY_PROMPT = """Role & Objective: You are an expert OCR TRANSCRIPTION assistant for digitized library and archival materials. Your PRIMARY job is a faithful, verbatim transcription of every word on this scanned page. Metadata fields on top (page_type, bibliographic, summaries) are secondary — transcription fidelity comes first.

Task: Extract all information from this single scanned page into the JSON structure below. body_text is the most important field and must contain the complete page text verbatim.

{
  "body_text": "<COMPLETE VERBATIM transcription of EVERY WORD OF PRINTED OR TYPED TEXT on the page, in reading order. Preserve original line breaks, paragraph breaks, spacing, spelling, punctuation, capitalization, and diacritics EXACTLY as they appear. If body_text ends up empty, you have failed the task.>",

  "page_type": "<book_page | title_page | table_of_contents | index | manuscript | sheet_music | newspaper | map | photograph | illustration | plate | form | correspondence | ephemera | blank | mixed>",

  "page_number": "<printed/stamped/handwritten page or folio number>",
  "header": "<running header text if present>",

  "bibliographic": {
    "title": "", "creator": "", "date": "", "language": "",
    "other_languages": [], "publisher": "", "place": ""
  },

  "structure": {
    "headings": [], "footnotes": [], "lists": [],
    "poetry_verse": "<verse text preserving line breaks, or null>"
  },

  "tables": [
    {"caption": "", "table_data": "<2D array or key-value object>"}
  ],

  "visual_elements": {
    "illustrations": [
      {"description": "<factual description, NO interpretation>", "caption": "", "position": ""}
    ],
    "musical_notation": {
      "present": false, "instrument_voice": "", "key_signature": "",
      "time_signature": "", "tempo_markings": "", "dynamics": [],
      "lyrics": "", "description": ""
    },
    "maps": {"present": false, "region": "", "description": ""}
  },

  "marginalia": [
    {"text": "<transcription or [illegible]>", "location": "", "writing_instrument": ""}
  ],

  "stamps_marks": [
    {"text": "", "type": "<library_stamp | bookplate | accession_number | call_number | barcode | ownership_mark | censor_mark | other>"}
  ],

  "physical_condition": {
    "damage": "", "legibility": "<fully_legible | mostly_legible | partially_legible | poor>",
    "scan_issues": ""
  },

  "one_sentence_summary": "",
  "confidence_percentage": <float 0-100>,
  "confidence_narrative": "",

  "continuation": {
    "continues_from_previous": false, "continues_to_next": false,
    "continuation_type": "<null | text | table | verse | musical_notation | list>"
  },

  "other_metadata": {}
}

TRANSCRIPTION RULES (HIGHEST PRIORITY):
- body_text MUST contain the full page text word-for-word. Do NOT summarize, paraphrase, or skip text.
- Preserve original spelling, punctuation, line breaks, diacritics, ligatures.
- For Fraktur/Gothic/blackletter: transcribe using modern Latin equivalents (ſ → s) but keep all other orthography.
- Illegible word → [illegible]. Uncertain → best guess [?]. Missing → [...].
- Marginalia and stamps go in their own fields, NOT body_text.
- TEXT WITHIN ILLUSTRATIONS still goes in body_text — illustrations describes the visual; readable words go in body_text regardless of styling.
- Title pages, covers, colophons: every legible word goes in body_text even if the page looks decorative.

Output ONLY valid JSON. No markdown fences, no introductory text."""

# Used when body_text comes back empty for a page that isn't blank/illustration/photograph/map.
LIBRARY_TRANSCRIPTION_RETRY_PROMPT = (
    "Transcribe every readable word on this scanned page in reading order "
    "(top-to-bottom, left-to-right). Include text in banners, decorative titles, "
    "captions, and frames — any text visible on the page, regardless of styling. "
    "Preserve spelling and diacritics exactly as printed. "
    'Output ONLY: {"body_text": "...full verbatim transcription here..."}'
)

print(f"Library prompt: {len(LIBRARY_PROMPT)} chars")
print(f"Retry prompt:   {len(LIBRARY_TRANSCRIPTION_RETRY_PROMPT)} chars")


## 4. `extract_page` — single-page VLM call with sliding-window context

For each page we send three images to the VLM: the **previous** page, the
**center** page (the one we want extracted), and the **next** page. The
prompt explicitly tells the model to extract from the center page only;
the surrounding pages help it set continuation flags correctly when a
chapter, table, or verse spills across pages.


In [ ]:
def extract_page(image, prompt, *, filename=None, links=None,
                 prev_image=None, next_image=None, max_tokens=VLLM_MAX_TOKENS):
    """Run a per-page library extraction with a 3-page sliding window.

    Role: one VLM call per page with its neighbors included as visual
    context. Builds the multi-image prompt — optional previous page
    (context only), the center page (the one to extract), optional
    next page (context only) — and tells the model in the prompt
    prefix which image is the target so it doesn't mix content
    across pages. Embedded hyperlinks (from `extract_links`) are
    surfaced as text. Returns the raw response string; the caller
    is expected to feed it through `parse_vlm_json`.
    """
    parts = []
    if filename:
        parts.append(f"Source filename: {filename}")
    if links:
        parts.append("Hyperlinks on this page:\n" +
                     "\n".join(f"  {l['text']} -> {l['url']}" for l in links))
    if prev_image or next_image:
        parts.append(
            "CONTEXT: Surrounding pages are shown for context only. "
            "Extract data from the CENTER PAGE only. Use the surrounding pages "
            "to set continuation flags when content spans page boundaries."
        )
    text = ("\n\n".join(parts) + "\n\n" + prompt) if parts else prompt

    content = []
    if prev_image:
        content.append({"type": "text", "text": "[PREVIOUS PAGE — context only]"})
        content.append({"type": "image", "image": _encode_image_b64(prev_image)})
    content.append({"type": "text", "text": "[CENTER PAGE — extract this page]"})
    content.append({"type": "image", "image": _encode_image_b64(image)})
    if next_image:
        content.append({"type": "text", "text": "[NEXT PAGE — context only]"})
        content.append({"type": "image", "image": _encode_image_b64(next_image)})
    content.append({"type": "text", "text": text})

    return run_vlm([{"role": "user", "content": content}], max_tokens)


print("extract_page ready.")


## 5. Pick a sample document and render its pages

Library scans are typically rendered at higher DPI than the grant docs —
3x scale (~216 DPI) — because preserving small glyphs (footnote marks,
diacritics, accents) matters more than tight token budgets here.


In [ ]:
import fitz

# Paths pick themselves from RUN_LOCATION at the top of the notebook.
# `laptop_vpn` looks for PDFs in `data/library/` at the repo root — create
# that folder and drop your sample PDFs in if you don't have it yet. Outputs
# from the batch step land in `<ocr_dir>/vlm_output/`.
OCR_DIRS = {
    "in_cluster": Path("/ocr"),
    "laptop_vpn": Path("../../data/library"),
}
ocr_dir = OCR_DIRS[RUN_LOCATION]
OUT_DIR = ocr_dir / "output"
OUT_DIR.mkdir(exist_ok=True)

pdfs = sorted(ocr_dir.glob("*.pdf"))
assert pdfs, f"No PDFs in {ocr_dir}/. Upload one first."

for p in pdfs:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

DOC_PATH = pdfs[0]
print(f"\nUsing: {DOC_PATH.name}")
print(f"Output dir: {OUT_DIR}")

doc = fitz.open(str(DOC_PATH))
page_images, page_links = [], []
for page in doc:
    pix = page.get_pixmap(matrix=fitz.Matrix(3.0, 3.0))
    page_images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    page_links.append(extract_links(page))
doc.close()
print(f"{len(page_images)} page(s) rendered")
display(page_images[0].resize((page_images[0].width // 3, page_images[0].height // 3)))


## 6. Extract a single page

Quick sanity check before kicking off the batch — extract the first page
and inspect what came back. If the page type, body_text length, and
bibliographic fields all look right, the prompt and the model are both
behaving.


In [ ]:
PAGE_IDX = 0
img = page_images[PAGE_IDX]
prev_img = page_images[PAGE_IDX - 1] if PAGE_IDX > 0 else None
next_img = page_images[PAGE_IDX + 1] if PAGE_IDX < len(page_images) - 1 else None

t0 = time.time()
raw = extract_page(img, LIBRARY_PROMPT, filename=DOC_PATH.name,
                   links=page_links[PAGE_IDX],
                   prev_image=prev_img, next_image=next_img)
print(f"Extraction: {time.time() - t0:.1f}s")

parsed, err = parse_vlm_json(raw)
assert parsed is not None, f"Parse failed: {err}"

bib = parsed.get("bibliographic", {}) or {}
print(f"\npage_type:    {parsed.get('page_type')}")
print(f"page_number:  {parsed.get('page_number') or '(none)'}")
print(f"language:     {bib.get('language') or '(unknown)'}")
print(f"title:        {bib.get('title') or '(none)'}")
print(f"creator:      {bib.get('creator') or '(none)'}")
print(f"summary:      {parsed.get('one_sentence_summary')}")
print(f"confidence:   {parsed.get('confidence_percentage')}%")
body = parsed.get("body_text") or ""
print(f"\nbody_text:    {len(body)} chars")
print("─" * 70)
print(body[:1500])
if len(body) > 1500:
    print("...")


## 7. Batch the whole folder

For each PDF: render → extract every page with the sliding window →
aggregate per-page results into a doc-level pass-1 JSON →
**pass 2** synthesizes catalog-style metadata across pages.

The pass-1 output is purely additive: per-page records aggregated by
field (all illustrations together, all stamps together, etc.) with no
cross-page mutation. Pass 2 is a single text-only VLM call that produces
the catalog metadata block.

We also have a small **transcription-retry** safety net: if a page comes
back with empty `body_text` and isn't a blank/photograph/map, we re-prompt
the model with a transcription-only prompt. This catches the common
failure mode of stylized title pages getting flagged as "decorative" and
returning nothing.


In [ ]:
SYNTHESIS_PROMPT = """You are given the per-page extraction results from a digitized library document.
Each page was processed independently. Your job is to produce ONLY a
document-level cataloging summary. Do NOT rewrite, merge, or modify the
per-page content — it is the source of truth.

Return a JSON object with catalog-style metadata:
{
  "document_title": "<best title found across pages, usually the title page>",
  "document_type": "<book | manuscript | periodical_issue | newspaper | sheet_music | atlas | map | correspondence | pamphlet | ephemera | mixed>",
  "creator": "", "creator_role": "", "date": "",
  "publisher": "", "place_of_publication": "",
  "primary_language": "", "other_languages": [],
  "subject_tags": ["<2-8 LoC-style subject tags>"],
  "document_summary": "<3-4 sentence summary synthesizing across ALL pages>",
  "collection_notes": {
    "call_number": "", "accession_number": "",
    "isbn_issn": "", "provenance": "<summary of stamps/bookplates/ownership marks seen>"
  },
  "physical_description_summary": "",
  "notable_features": ["<illustrations, musical notation, marginalia, multilingual sections, missing pages, etc.>"],
  "cross_page_notes": ["<observations about content spanning pages>"]
}

Output ONLY valid JSON. No markdown fences."""


In [ ]:
import collections

# Page types where empty body_text is legitimate — skip the retry on these.
_NO_BODY_TEXT_TYPES = {"blank", "photograph", "map", "illustration", "plate"}


def assemble_pass1(filename: str, page_results: list, model_name: str) -> dict:
    """Aggregate per-page VLM output into one doc-level JSON.

    Pass 1 is strictly additive — no cross-page merging. We pick the best
    bibliographic value across pages, concatenate body_text with [PAGE N]
    markers, dedup stamps by (text, type), and gather illustrations,
    marginalia, headings, etc. with their page references.
    """
    from datetime import datetime

    def _best(field):
        for pr in page_results:
            v = pr.get("extracted", {}).get("bibliographic", {}).get(field)
            if v and str(v).strip():
                return v
        return ""

    bib = {k: _best(k.lower()) for k in
           ("Title", "Creator", "Date", "Language", "Publisher", "Place")}

    body_parts, page_types, summaries = [], [], []
    headings, footnotes, illustrations = [], [], []
    marginalia, stamps_raw, musical, maps_, tables_ = [], [], [], [], []
    confidences, page_confidences, page_flags = [], [], {}

    for pr in page_results:
        pg = pr["page"]
        d = pr.get("extracted", {})
        page_types.append({"PageNumber": pg, "PageType": d.get("page_type", "unknown")})
        summaries.append({"PageNumber": pg, "Summary": d.get("one_sentence_summary") or "N/A"})
        if d.get("body_text"):
            body_parts.append(f"[PAGE {pg}]\n{d['body_text']}")
        for h in d.get("structure", {}).get("headings") or []:
            if h: headings.append({"PageNumber": pg, "Heading": h})
        for fn in d.get("structure", {}).get("footnotes") or []:
            if fn: footnotes.append({"PageNumber": pg, "Footnote": fn})
        for ill in d.get("visual_elements", {}).get("illustrations") or []:
            illustrations.append({"PageNumber": pg, **{k: ill.get(k.lower(), "")
                for k in ("Description", "Caption", "Position")}})
        mn = d.get("visual_elements", {}).get("musical_notation") or {}
        if mn.get("present"):
            musical.append({"PageNumber": pg, **mn})
        mp = d.get("visual_elements", {}).get("maps") or {}
        if mp.get("present"):
            maps_.append({"PageNumber": pg, **mp})
        for m in d.get("marginalia") or []:
            if m.get("text"):
                marginalia.append({"PageNumber": pg, **m})
        for s in d.get("stamps_marks") or []:
            if s.get("text"):
                stamps_raw.append({"PageNumber": pg, **s})
        for t in d.get("tables") or []:
            tables_.append({"PageNumber": pg, **t})

        conf = d.get("confidence_percentage")
        if isinstance(conf, (int, float)):
            confidences.append(conf)
        page_confidences.append({
            "PageNumber": pg, "ConfidencePercentage": conf or 0,
            "ConfidenceNarrative": d.get("confidence_narrative") or "",
        })
        flags = {k: v for k, v in d.items()
                 if k in ("body_text_repetition_detected",
                          "body_text_empty_after_retry",
                          "body_text_source", "parse_error") and v}
        if flags:
            page_flags[pg] = flags

    # Dedup stamps by (text, type), accumulate the page list per stamp.
    stamps = {}
    for s in stamps_raw:
        key = (s["text"], s.get("type", "other"))
        stamps.setdefault(key, {"Text": s["text"], "Type": s.get("type", "other"), "Pages": []})
        stamps[key]["Pages"].append(s["PageNumber"])

    return {
        "Filename": filename,
        "PageCount": len(page_results),
        "RunInfo": {
            "Model": model_name,
            "Timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
        },
        "ConfidencePercentage": round(sum(confidences) / len(confidences), 1) if confidences else 0,
        "PageConfidences": page_confidences,
        "PageFlags": page_flags,
        "PageResults": page_results,
        "Bibliographic": bib,
        "PageTypes": page_types,
        "OneSentenceNarrativeSummary": summaries,
        "BodyText": "\n\n".join(body_parts),
        "Headings": headings,
        "Footnotes": footnotes,
        "TablesCollection": tables_,
        "Illustrations": illustrations,
        "MusicalNotation": musical,
        "Maps": maps_,
        "Marginalia": marginalia,
        "StampsMarks": list(stamps.values()),
    }


def run_pass2(page_results: list) -> dict | None:
    """Run pass 2: ask the VLM for catalog-style metadata across all pages.

    Role: the second VLM call per document. Sends every page's parsed
    JSON (text only, no images) and asks for catalog metadata —
    title, type, creator, date, subject tags, document_summary,
    cross_page_notes. Output is strictly additive: the per-page
    records remain untouched. Returns the parsed dict, or None on
    failure.
    """
    parts = []
    for pr in page_results:
        parts.append(f"--- PAGE {pr['page']} ---")
        parts.append(json.dumps(pr.get("extracted", {}), indent=1, ensure_ascii=False))
    body = "\n".join(parts)
    msgs = [{"role": "user", "content": [
        {"type": "text", "text": f"{SYNTHESIS_PROMPT}\n\n{body}"}
    ]}]
    raw = run_vlm(msgs, max_tokens=1500)
    parsed, _ = parse_vlm_json(raw)
    return parsed


def repetition_loop_detected(body_text: str) -> tuple[bool, str, int]:
    """Detect VLM generation loops by counting repeated lines in body_text.

    Role: cheap quality guard before we ship the page result. The VLM
    occasionally gets stuck emitting the same line over and over
    inside a single body_text string — the JSON parses fine but the
    content is garbage. If any one line shows up 5+ times we declare
    the page looped so the caller can clear body_text and fall back
    to the transcription-retry path.

    Returns ``(is_looped, most_common_line, count)``.
    """
    lines = [l.strip() for l in (body_text or "").splitlines() if l.strip()]
    if not lines:
        return False, "", 0
    line, count = collections.Counter(lines).most_common(1)[0]
    return count >= 5, line, count


print("Helpers ready.")


In [ ]:
# ocr_dir + OUT_DIR were set above based on RUN_LOCATION.
SKIP_EXISTING = True

pdf_files = sorted(ocr_dir.glob("*.pdf"))
print(f"{len(pdf_files)} PDF(s) to process")

for pdf_path in pdf_files:
    out_path = OUT_DIR / f"{pdf_path.stem}_extracted.json"
    final_path = OUT_DIR / f"{pdf_path.stem}_final.json"

    if SKIP_EXISTING and final_path.exists():
        print(f"SKIP (already done): {pdf_path.name}")
        continue

    print(f"\n=== {pdf_path.name} ===")
    doc = fitz.open(str(pdf_path))
    images, links = [], []
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(3.0, 3.0))
        images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
        links.append(extract_links(page))
    doc.close()
    print(f"  {len(images)} page(s) rendered")

    page_results = []
    for i, img in enumerate(images):
        page_num = i + 1
        prev_img = images[i - 1] if i > 0 else None
        next_img = images[i + 1] if i < len(images) - 1 else None

        t0 = time.time()
        raw = extract_page(img, LIBRARY_PROMPT, filename=pdf_path.name,
                           links=links[i], prev_image=prev_img, next_image=next_img)
        elapsed = time.time() - t0

        extracted, err = parse_vlm_json(raw)
        if extracted is None:
            print(f"  page {page_num}: PARSE FAILED ({err})")
            extracted = {"parse_error": err, "confidence_percentage": 0}

        # Repetition guard — clear body_text if we caught a generation loop.
        looped, line, count = repetition_loop_detected(extracted.get("body_text") or "")
        if looped:
            print(f"  page {page_num}: repetition loop ({line[:30]!r} x{count}) — clearing body_text")
            extracted["body_text"] = ""
            extracted["body_text_repetition_detected"] = True

        # Transcription retry — if body_text is empty for a page that should have text.
        body = (extracted.get("body_text") or "").strip()
        page_type = (extracted.get("page_type") or "").lower().strip()
        if not body and page_type not in _NO_BODY_TEXT_TYPES and not extracted.get("parse_error"):
            print(f"  page {page_num}: body_text empty ({page_type=!r}) — retry")
            retry_raw = extract_page(img, LIBRARY_TRANSCRIPTION_RETRY_PROMPT,
                                     filename=pdf_path.name)
            retry_parsed, _ = parse_vlm_json(retry_raw)
            retry_text = (retry_parsed or {}).get("body_text", "").strip() or retry_raw.strip()
            if retry_text:
                extracted["body_text"] = retry_text
                extracted["body_text_source"] = "transcription_retry"
            else:
                extracted["body_text_empty_after_retry"] = True

        page_results.append({"page": page_num, "method": "vlm_image",
                             "elapsed_ms": round(elapsed * 1000, 1),
                             "extracted": extracted})
        print(f"  page {page_num}/{len(images)}: {elapsed:.1f}s "
              f"({len(extracted.get('body_text') or '')} body chars)")

    pass1 = assemble_pass1(pdf_path.name, page_results, MODEL_NAME)
    out_path.write_text(json.dumps(pass1, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"  pass 1 saved: {out_path.name}")

    t2 = time.time()
    synthesis = run_pass2(page_results)
    if synthesis:
        final = {**pass1, "DocumentSynthesis": synthesis}
        final_path.write_text(json.dumps(final, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(f"  pass 2 saved: {final_path.name} ({time.time() - t2:.1f}s) — "
              f"{synthesis.get('document_type', '?')}: {synthesis.get('document_title', '?')[:60]}")
    else:
        print(f"  pass 2 FAILED — keeping pass 1 only")

print("\nBatch complete.")


## 8. Inspect a result

Look at the doc-level JSON. Useful fields:

- `DocumentSynthesis.document_title` / `document_type` / `creator` /
  `document_summary` — pass-2 catalog metadata
- `Bibliographic` — best-of bibliographic fields across pages
- `BodyText` — full transcription with `[PAGE N]` markers
- `StampsMarks`, `Marginalia`, `Illustrations` — provenance / paratext
- `PageFlags` — pages where the retry path fired or repetition was caught


In [ ]:
out_files = sorted(OUT_DIR.glob("*_final.json"))
assert out_files, f"No outputs in {OUT_DIR}/. Run the batch cell first."
sample = json.loads(out_files[0].read_text(encoding="utf-8"))

synth = sample.get("DocumentSynthesis", {}) or {}
print(f"=== {sample['Filename']} ===")
print(f"type:        {synth.get('document_type', '?')}")
print(f"title:       {synth.get('document_title', '?')}")
print(f"creator:     {synth.get('creator', '?')}")
print(f"date:        {synth.get('date', '?')}")
print(f"languages:   {synth.get('primary_language', '?')} "
      f"(also: {', '.join(synth.get('other_languages') or []) or 'none'})")
print(f"tags:        {', '.join(synth.get('subject_tags') or [])}")
print(f"\nsummary:\n  {synth.get('document_summary', '?')}")
print(f"\npage count:  {sample.get('PageCount')}")
print(f"body chars:  {len(sample.get('BodyText') or '')}")
print(f"headings:    {len(sample.get('Headings') or [])}")
print(f"stamps:      {len(sample.get('StampsMarks') or [])}")
print(f"marginalia:  {len(sample.get('Marginalia') or [])}")
print(f"illustrations: {len(sample.get('Illustrations') or [])}")
flags = sample.get("PageFlags") or {}
if flags:
    print(f"\npages with extraction flags: {list(flags.keys())}")
